In [1]:
import numpy as np, pandas as pd
from scipy import stats
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(0)
d, b0, K, R, eps, alpha = 10, -3.8, 30, 20, 0.001, 0.05
theta_bar = rng.normal(0, 0.5, d)
hs = [0.0, 0.1, 0.2, 0.4, 0.8]
ns = [500, 2000, 8000, 30000]

hs_sum = pd.read_csv("../docs/hospital_summary_2022.csv")
logn = np.log(hs_sum["n"].clip(lower=50))
mu, sig = logn.mean(), logn.std()

def gen(theta, n):
    X = rng.normal(size=(n, d))
    y = (rng.random(n) < 1 / (1 + np.exp(-(b0 + X @ theta)))).astype(int)
    return X, y

def ll(p, y):
    p = np.clip(p, 1e-4, 1 - 1e-4)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))

def unit():
    u = rng.normal(size=d); return u / np.linalg.norm(u)

rows = []
for h in hs:
    for r in range(R):
        sizes = np.clip(np.exp(rng.normal(mu, sig, K)), 500, 50000).astype(int)
        data = [gen(theta_bar + h * unit(), n) for n in sizes]
        Xp = np.vstack([a for a, _ in data]); yp = np.concatenate([b for _, b in data])
        pooled = LogisticRegression(max_iter=300).fit(Xp, yp)
        th_f = theta_bar + h * unit()
        Xt, yt = gen(th_f, 100000)
        l_fed_true = ll(pooled.predict_proba(Xt)[:, 1], yt).mean()
        for n in ns:
            Xf, yf = gen(th_f, n)
            a, c = slice(0, n // 2), slice(n // 2, n // 2 + n // 4)
            if yf[a].sum() >= 3:
                loc = LogisticRegression(max_iter=300).fit(Xf[a], yf[a])
                p_loc_t = loc.predict_proba(Xt)[:, 1]; p_loc_c = loc.predict_proba(Xf[c])[:, 1]
            else:
                m = max(yf[a].mean(), 1e-3); p_loc_t = np.full(len(yt), m); p_loc_c = np.full(len(yf[c]), m)
            true_diff = l_fed_true - ll(p_loc_t, yt).mean()
            dd = ll(pooled.predict_proba(Xf[c])[:, 1], yf[c]) - ll(p_loc_c, yf[c])
            ub = dd.mean() + stats.norm.ppf(1 - alpha) * dd.std(ddof=1) / np.sqrt(len(dd))
            cert = ub <= eps
            rows.append((h, n, true_diff <= eps, cert, cert and true_diff > eps))

sim = pd.DataFrame(rows, columns=["h", "n", "true_no_harm", "certified", "false_cert"])
sim.to_csv("../outputs/tables/sim_phase_map_v1.csv", index=False)
for col, title in [("true_no_harm", "P(pooled model truly causes no harm)"),
                   ("certified", "P(certificate approves pooled model)"),
                   ("false_cert", "P(approved but truly harmful)")]:
    print(title)
    print(sim.pivot_table(index="h", columns="n", values=col, aggfunc="mean").round(2), "\n")

P(pooled model truly causes no harm)
n    500    2000   8000   30000
h                              
0.0    1.0   1.00    1.0   1.00
0.1    1.0   1.00    1.0   1.00
0.2    1.0   1.00    1.0   0.95
0.4    1.0   0.90    0.2   0.00
0.8    0.6   0.05    0.0   0.00 

P(certificate approves pooled model)
n    500    2000   8000   30000
h                              
0.0   0.15   0.45   0.50   1.00
0.1   0.15   0.15   0.50   0.90
0.2   0.20   0.25   0.20   0.35
0.4   0.20   0.10   0.15   0.00
0.8   0.05   0.00   0.00   0.00 

P(approved but truly harmful)
n    500    2000   8000   30000
h                              
0.0    0.0    0.0    0.0    0.0
0.1    0.0    0.0    0.0    0.0
0.2    0.0    0.0    0.0    0.0
0.4    0.0    0.0    0.0    0.0
0.8    0.0    0.0    0.0    0.0 



In [2]:
rng = np.random.default_rng(1)
R2 = 40
blends = [0.25, 0.5, 0.75, 1.0]
rows = []
for h in hs:
    for r in range(R2):
        sizes = np.clip(np.exp(rng.normal(mu, sig, K)), 500, 50000).astype(int)
        data = [gen(theta_bar + h * unit(), n) for n in sizes]
        Xp = np.vstack([a for a, _ in data]); yp = np.concatenate([b for _, b in data])
        pooled = LogisticRegression(max_iter=300).fit(Xp, yp)
        th_f = theta_bar + h * unit()
        Xt, yt = gen(th_f, 100000)
        pf_t = pooled.predict_proba(Xt)[:, 1]
        for n in ns:
            Xf, yf = gen(th_f, n)
            a, c = slice(0, n // 2), slice(n // 2, n // 2 + n // 4)
            if yf[a].sum() >= 3:
                loc = LogisticRegression(max_iter=300).fit(Xf[a], yf[a])
                pl_t = loc.predict_proba(Xt)[:, 1]; pl_c = loc.predict_proba(Xf[c])[:, 1]
            else:
                m = max(yf[a].mean(), 1e-3); pl_t = np.full(len(yt), m); pl_c = np.full(len(yf[c]), m)
            pf_c = pooled.predict_proba(Xf[c])[:, 1]
            base_t = ll(pl_t, yt).mean(); base_c = ll(pl_c, yf[c])
            sel = 0.0
            for a_b in blends:                                  # fixed sequence, local to pooled
                dd = ll((1 - a_b) * pl_c + a_b * pf_c, yf[c]) - base_c
                ub = dd.mean() + stats.norm.ppf(1 - alpha) * dd.std(ddof=1) / np.sqrt(len(dd))
                if ub <= eps: sel = a_b
                else: break
            true_sel = ll((1 - sel) * pl_t + sel * pf_t, yt).mean() - base_t
            true_pool = ll(pf_t, yt).mean() - base_t
            rows.append((h, n, true_pool > eps, sel > 0, true_sel > eps, -true_sel * 1000, -true_pool * 1000))

s2 = pd.DataFrame(rows, columns=["h", "n", "pooled_harmed", "nonlocal", "selected_harmed", "gain_sel_x1000", "gain_pooled_x1000"])
s2.to_csv("../outputs/tables/sim_certified_selection_v1.csv", index=False)
for col, title in [("pooled_harmed", "P(always-pooled harms the hospital)"),
                   ("selected_harmed", "P(certified choice harms the hospital)  [must be <= 0.05]"),
                   ("nonlocal", "P(certified choice is not local)"),
                   ("gain_sel_x1000", "mean true loss gain of certified choice over local (x1000)"),
                   ("gain_pooled_x1000", "mean true loss gain of always-pooled over local (x1000)")]:
    print(title)
    print(s2.pivot_table(index="h", columns="n", values=col, aggfunc="mean").round(2), "\n")

P(always-pooled harms the hospital)
n    500    2000   8000   30000
h                              
0.0    0.0   0.00   0.00   0.00
0.1    0.0   0.00   0.00   0.00
0.2    0.0   0.00   0.02   0.05
0.4    0.0   0.12   0.85   1.00
0.8    0.3   0.95   1.00   1.00 

P(certified choice harms the hospital)  [must be <= 0.05]
n    500    2000   8000   30000
h                              
0.0   0.00   0.00   0.00   0.00
0.1   0.00   0.00   0.00   0.00
0.2   0.00   0.00   0.00   0.00
0.4   0.00   0.00   0.08   0.00
0.8   0.02   0.02   0.02   0.02 

P(certified choice is not local)
n    500    2000   8000   30000
h                              
0.0   0.45   0.90   1.00    1.0
0.1   0.45   0.82   1.00    1.0
0.2   0.42   0.68   1.00    1.0
0.4   0.35   0.57   0.78    1.0
0.8   0.25   0.22   0.15    0.1 

mean true loss gain of certified choice over local (x1000)
n    500    2000   8000   30000
h                              
0.0   8.88   4.17   1.25   0.28
0.1  10.49   4.51   0.89   0.14
0.2   9.